In [1]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv("../../datasets/LI-Medium_Trans.csv")
trans_df.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:15,20,800104D70,20,800104D70,8095.07,US Dollar,8095.07,US Dollar,Reinvestment,0
1,2022/09/01 00:18,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:23,1208,80010E430,1208,80010E430,2654.22,US Dollar,2654.22,US Dollar,Reinvestment,0
3,2022/09/01 00:19,3203,80010EA80,3203,80010EA80,13284.41,US Dollar,13284.41,US Dollar,Reinvestment,0
4,2022/09/01 00:27,20,800104D20,20,800104D20,9.72,US Dollar,9.72,US Dollar,Reinvestment,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/27 14:58]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Account
805AB2160    2
805CAAD20    2
806E32CA0    2
806E3F3E0    2
81906AAA0    2
8191E6CA0    2
81F798CD0    2
81F79BC90    2
82CA90C40    2
82CA983A0    2
82E0FFB60    2
82E104890    2
8304B3520    2
8304B8850    2
83A0A2180    2
83A0A3750    2
83CBE9F20    2
83CBF2D40    2
843960ED0    2
843970320    2
843D413B0    2
843D4A420    2
84538D7C0    2
845391C30    2
845ED8110    2
845ED8160    2
847134CE0    2
84718C870    2
84A2B5120    2
84A2B9F50    2
84C504ED0    2
84C505B70    2
84FE18D70    2
84FE18E10    2
Name: Bank, dtype: int64

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 11839293


In [5]:
# Analyze accounts.
accounts_df = pd.read_csv("../../datasets/LI-Medium_accounts.csv")
print("SIZE:", accounts_df.shape[0])
accounts_df.head(5)

SIZE: 2040823


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Bank of New Orleans,9778,834A31900,2AA23456AF0,Corporation #398
1,First Bank of Indianapolis,6522,833FCBD00,2AA23339000,Sole Proprietorship #29247
2,Savings Bank of Danbury,3138765,834EA6680,2AA2376A660,Partnership #6612
3,First Bank of Laramie,31356,8023D3E00,2AA2362E530,Corporation #2749
4,Savings Bank of Providence,3132123,83C4D6D80,2AA236E07C0,Partnership #5472


In [6]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 4114338


In [7]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return  unique_account_size > 5 and unique_account_size < 10
ranged_trans_usd_sept_df = trans_usd_sept_1st_df.groupby(["From Bank", "Account"]).filter(filter_function)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 651567


In [8]:
#1. Cuenta de origen, cuenta de destino y monto para transacciones USD menores a 50.

low_profile_transactions = trans_usd_df[trans_usd_df["Amount Paid"] < 50].copy()
low_profile_transactions = low_profile_transactions[["From Bank", "Account", "To Bank", "Account.1", "Amount Paid"]]
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
4,20,800104D20,20,800104D20,9.72
5,20,800104D70,20,800104D70,5.38
6,1208,80010E430,1208,80010E430,7.66
7,11,80010E600,11,80010E600,16.33
8,1208,80010E650,1208,80010E650,4.86
...,...,...,...,...,...
31251310,145434,8194D9AA1,145434,8194D9AA0,0.05
31251312,145434,8194D9AA1,145434,8194D9AA0,0.20
31251328,241905,81AAB09E1,241905,81AAB09E0,0.18
31251330,241905,81AAB09E1,241905,81AAB09E0,0.04


In [9]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
bank_name_by_bank_df = accounts_df[["Bank ID", "Bank Name"]].drop_duplicates(subset=["Bank ID"])
max_amount_bank = max_amount_trans_usd.merge(bank_name_by_bank_df, left_on="From Bank", right_on="Bank ID", how="left")
max_amount_bank["Bank Name"] = max_amount_bank.apply(lambda row: row["Bank Name"] if pd.notna(row["Bank Name"]) and str(row["Bank Name"]).strip() != "" else f"UNKNOWN_{int(row['From Bank'])}", axis=1)
max_amount_bank[["From Bank", "Account", "Bank Name", "Amount Paid"]]

,From Bank,Account,Bank Name,Amount Paid
0,0,803C88B10,National Bank of Fort Wayne,1.567364e+09
1,1,8001464D0,Italy Bank #68,3.350756e+08
2,2,80008B620,China Bank #51,1.902725e+08
3,3,8000801B0,Germany Bank #85,1.217279e+08
4,4,80C2E0B10,Japan Bank #18,2.030984e+04
...,...,...,...,...
46640,3219745,8505E6FA0,Savings Bank of Helena,5.051400e+02
46641,3219749,8505EAF30,National Bank of Providence,6.269670e+03
46642,3219753,8505F16E0,Savings Bank of Philadelphia,1.458200e+02
46643,3219754,8505F24F0,First Bank of Indianapolis,2.381830e+03


In [10]:
#3. Cuenta de origen y monto de transacciones USD en [2022-09-06, 2022-09-15]
# con monto < 1% del promedio para el mismo Payment Format en [2022-09-01, 2022-09-05].

# Usamos límites por día inclusivos con ventanas [start, next_day) para no perder registros con hora.
trans_usd_base_period_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/01") & (trans_usd_df["Timestamp"] < "2022/09/06")]
trans_usd_eval_period_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/06") & (trans_usd_df["Timestamp"] < "2022/09/16")]

avg_amounts_per_type = trans_usd_base_period_df.groupby(["Payment Format"], as_index=False)["Amount Paid"].mean().rename(columns={"Amount Paid": "AVG"})

trans_usd_eval_with_avg_df = trans_usd_eval_period_df.merge(avg_amounts_per_type, on=["Payment Format"], how="inner")

lower_trans_usd_eval_with_avg_df = trans_usd_eval_with_avg_df[
    trans_usd_eval_with_avg_df["Amount Paid"] < trans_usd_eval_with_avg_df["AVG"] * 0.01
]

lower_trans_usd_eval_with_avg_df[["From Bank", "Account", "Amount Paid"]]

,From Bank,Account,Amount Paid
0,3,80F353730,13666.55
1,16112,818A4BDA0,11430.74
2,211013,811B48C60,3196.05
3,211013,811B48C60,6416.59
4,211013,811B48C60,1469.93
...,...,...,...
6604430,225892,835236330,181.45
6604431,225892,835236330,529.40
6604432,225892,835236330,213.43
6604434,68444,84194EDA1,0.19


In [18]:
#4. Cuentas que cumplen patrón scatter-gather con >= 5 cuentas intermedias
# distintas por par (A, B), para cuentas que enviaron en USD en [2022-09-01, 2022-09-05].

trans_usd_scatter_window_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/01") & (trans_usd_df["Timestamp"] < "2022/09/06")]

edges_df = trans_usd_scatter_window_df[["From Bank", "Account", "To Bank", "Account.1"]].drop_duplicates()

source_fanout_df = edges_df.groupby(["From Bank", "Account"]).size().reset_index(name="distinct_destinations")
candidate_sources_df = source_fanout_df[source_fanout_df["distinct_destinations"] >= 5][["From Bank", "Account"]]

first_hop_df = edges_df.merge(candidate_sources_df, on=["From Bank", "Account"], how="inner").rename(columns={
    "From Bank": "Source Bank",
    "Account": "Source Account",
    "To Bank": "Interm Bank",
    "Account.1": "Interm Account",
})

second_hop_df = edges_df.rename(columns={
    "From Bank": "Interm Bank",
    "Account": "Interm Account",
    "To Bank": "Final Bank",
    "Account.1": "Final Account",
})

paths_df = first_hop_df.merge(second_hop_df, on=["Interm Bank", "Interm Account"], how="inner")
paths_df = paths_df[(paths_df["Source Bank"] != paths_df["Final Bank"]) | (paths_df["Source Account"] != paths_df["Final Account"])]

paths_df = paths_df.copy()
paths_df["interm_key"] = paths_df["Interm Bank"].astype(str) + "_" + paths_df["Interm Account"]

interm_count_df = paths_df.groupby(
    ["Source Bank", "Source Account", "Final Bank", "Final Account"]
)["interm_key"].nunique().reset_index(name="n_intermediaries")

qualified_pairs_df = interm_count_df[interm_count_df["n_intermediaries"] >= 5][
    ["Source Bank", "Source Account", "Final Bank", "Final Account"]
]

scatter_gather_pairs_df = qualified_pairs_df.rename(columns={
    "Source Bank": "From Bank",
    "Source Account": "Account",
    "Final Bank": "To Bank",
    "Final Account": "Account.1",
}).reset_index(drop=True)
scatter_gather_pairs_df

,From Bank,Account,To Bank,Account.1
0,214,8105424E0,9095,810542A20
1,214,8105424E0,20542,8105417C0
2,214,8105424E0,242350,8105425D0
3,741,80BD382D0,24545,80BD68440
4,741,80BD382D0,25733,80BD5EFC0
...,...,...,...,...
243,226649,80BD67FD0,226696,80BD8F860
244,230096,80BD5E5E0,24545,80BD68440
245,230096,80BD5E5E0,25733,80BD5EFC0
246,230096,80BD5E5E0,124526,80BD36380


In [19]:
#5. Cantidad de transacciones del período [2022-09-01, 2022-09-05]
# con formato Wire o ACH cuyo monto convertido a USD sea menor a 1.
import json

# Only currencies present in cotizaciones.json; absent ones (Bitcoin) → NaN → discarded
CURRENCY_NAME_TO_CODE = {
    "Australian Dollar": "AUD",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Euro":              "EUR",
    "Mexican Peso":      "MXN",
    "Rupee":             "INR",
    "Ruble":             "RUB",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "UK Pound":          "GBP",
    "US Dollar":         None,   # base currency → rate = 1
    "Yen":               "JPY",
    "Yuan":              "CNY",
}

with open("../../datasets/cotizaciones.json") as f:
    cotizaciones_list = json.load(f)

# Rebuild { "YYYY-MM-DD": { "CODE": float } } from the flat list
raw_rates = {}
for entry in cotizaciones_list:
    raw_rates.setdefault(entry["date"], {})[entry["quote"]] = entry["rate"]

available_dates = sorted(raw_rates.keys())

def rates_for_date(date_str):
    """Most recent available rates on or before date_str (handles weekends)."""
    candidates = [d for d in available_dates if d <= date_str]
    return raw_rates[candidates[-1] if candidates else available_dates[0]]

# Build a tidy rates DataFrame: Date × Payment Currency → usd_rate
# JSON convention: 1 USD = X units of currency → 1 unit = 1/X USD
date_range = pd.date_range("2022-09-01", "2022-09-05", freq="D")
rates_rows = []
for dt in date_range:
    date_str = dt.strftime("%Y-%m-%d")
    day = rates_for_date(date_str)
    for name, code in CURRENCY_NAME_TO_CODE.items():
        usd_rate = 1.0 if code is None else 1.0 / day[code]
        rates_rows.append({"Date": date_str, "Payment Currency": name, "usd_rate": usd_rate})

rates_df = pd.DataFrame(rates_rows)

trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= "2022/09/01") & (trans_df["Timestamp"] < "2022/09/06")]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[
    (trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")
]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df["Date"] = (
    pd.to_datetime(trans_sept_1st_wire_or_ach_converted_df["Timestamp"]).dt.strftime("%Y-%m-%d")
)
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_converted_df.merge(
    rates_df, on=["Date", "Payment Currency"], how="left"
)
trans_sept_1st_wire_or_ach_converted_df["Amount"] = (
    trans_sept_1st_wire_or_ach_converted_df["Amount Paid"] * trans_sept_1st_wire_or_ach_converted_df["usd_rate"]
)
trans_sept_1st_wire_or_ach_lt_1_df = trans_sept_1st_wire_or_ach_converted_df[
    trans_sept_1st_wire_or_ach_converted_df["Amount"] < 1
]
print("SIZE:", trans_sept_1st_wire_or_ach_lt_1_df.shape[0])

SIZE: 26395
